# สร้าง Model คัดแยก SMS

### นิยามความหมายของ `group` และ `sub_group`

**`group` (กลุ่มหลัก — 2 คลาส):**
- **`positive`** (861 แถว): ข้อความ SMS ปกติ / ข้อความจากระบบหรือองค์กรที่มีตัวตนจริง
- **`negative`** (462 แถว): ข้อความ SMS หลอกลวง / สแปม / มิจฉาชีพ

**`sub_group` (กลุ่มย่อย — 8 คลาส):**

| `group` | `sub_group` | จำนวน | ความหมาย |
| :--- | :--- | ---: | :--- |
| `positive` | `promotion` | 532 | SMS โปรโมชันสินค้า สิทธิพิเศษ พอยท์ และบริการทางการค้า |
| `positive` | `carrier_vas` | 151 | SMS บริการเสริมจากค่ายมือถือ (เช่น ดูดวง ริงโทน กด `*...#`) |
| `positive` | `otp` | 70 | SMS รหัส OTP / รหัสยืนยันตัวตนในการทำรายการ |
| `positive` | `service_notice` | 63 | SMS แจ้งเตือนบริการ สถานะพัสดุ ระบบโทรศัพท์ และประกาศภาครัฐ |
| `positive` | `payment_alert` | 45 | SMS แจ้งชำระเงิน ยอดค้างชำระ เงินเข้า-ออก และธุรกรรมทางการเงิน |
| `negative` | `invite` | 293 | SMS ชักชวนเว็บพนัน แจกเครดิตฟรี รางวัลปลอม กู้เงินเถื่อน และงานออนไลน์ |
| `negative` | `phishing_link` | 129 | SMS แนบลิงก์ฟิชชิ่ง ปลอมเป็นธนาคาร ขนส่ง หรือหน่วยงานรัฐเพื่อขโมยข้อมูล |
| `negative` | `threat` | 40 | SMS ข่มขู่ / แจ้งเตือนเท็จ (เช่น พัสดุถูกอายัด ตัดไฟ หมายเรียก) เพื่อกดดันให้รีบดำเนินการ |

### install & import libarry

In [ ]:
# !pip install pandas

import re, math, unicodedata, itertools
import pandas as pd
from IPython.display import display
from functools import lru_cache
from pythainlp.tokenize import word_tokenize
from pythainlp.util import normalize as th_normalize
from pythainlp.corpus import thai_words, thai_stopwords
from pythainlp.corpus.tnc import word_freqs
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

### รวม Function Visualization
จุดที่ต้องการใส่ Visualization ให้ประการ cell code ไว้ล่างเซลนี้ โดย 1 Function ต่อ 1 cell

### ดึง Data ที่เตรียมไว้จาก data_set.csv

In [2]:
df = pd.read_csv("data_set.csv", encoding="utf-8-sig")

print(f"จำนวนข้อมูลทั้งหมด: {len(df)} แถว")
print("\nสัดส่วน group:")
print(df["group"].value_counts())
display(df.head())

จำนวนข้อมูลทั้งหมด: 1447 แถว

สัดส่วน group:
group
positive    893
negative    554
Name: count, dtype: int64


,id,sentence_raw,group,sub_group,source
0,1,เปิดดวงรับโชค ลุ้น 300 บาท กด *451# สมัครดวงรา...,positive,carrier_vas,tonkla_phone
1,2,ขอบคุณที่ชำระ 535 บ. 01/03/2024 <URL>,positive,payment_alert,Zoleir/Fraud-SMS-Detection-Application
2,3,Lazada เซลใหญ่วันเกิด แบรนด์ดังลดแรง สูงสุด 90...,positive,promotion,Zoleir/Fraud-SMS-Detection-Application
3,4,เมษานี้เย็นใจกับแป้นพิมพ์มือถือ Emerald Thai W...,positive,carrier_vas,ter_phone
4,5,1-20 เม.ย.68 ร่วมตอบแบบสอบถามสำมะโนประชากรและเ...,positive,gov_announcement,ter_phone


## NLP

### Normalize data ไม่สำคัญให้อยู่ในรูป <TAG>

In [3]:
# รูปแบบของ <TAG>
URL_RE = re.compile(
    r"(?:https?://[^\s<>\"']+|(?:www\.|lin\.ee/|cutt\.ly/|bit\.ly/|shorturl\.at/|urlzs\.com/|m\.ais\.co\.th/|s\.true\.th/)[A-Za-z0-9._~:/?#\[\]@!$&'()*+,;=%-]+|\b[A-Za-z0-9-]+\.(?:com|net|org|me|vip|fun|xyz|club|cc|info|top|site|online|shop|bet|win|live|asia|co\.th)(?:/[^\s<>\"']*)?)",
    re.IGNORECASE,
)
LINE_RE = re.compile(r"(?:(?:ไลน์|line|id|ไอดี)\s*[:：]?\s*)?@[A-Za-z0-9._-]{3,}", re.IGNORECASE)
USSD_RE = re.compile(r"\*\d+(?:\*\d+)*#|\b\d{4,6}#")
DATE_RE = re.compile(
    r"\b\d{1,2}/\d{1,2}/\d{2,4}\b|\d{1,2}\s*(?:ม\.ค\.|ก\.พ\.|มี\.ค\.|เม\.ย\.|พ\.ค\.|มิ\.ย\.|ก\.ค\.|ส\.ค\.|ก\.ย\.|ต\.ค\.|พ\.ย\.|ธ\.ค\.)\s*\d{2,4}"
)
TIME_RE = re.compile(r"\b\d{1,2}[:.]\d{2}\s*น\.")
OTP_RE = re.compile(
    r"<OTP\s+\d{4,8}>|\bOTP\s*(?:is|=|:)?\s*\d{4,8}\b|\b\d{6}(?=\s+is your\b)",
    re.IGNORECASE,
)
OTP_TH_RE = re.compile(r"(รหัส(?:ยืนยัน|ผ่านครั้งเดียว|รักษาความปลอดภัย)[^\d\n]{0,15})\b\d{4,8}\b")
REF_RE = re.compile(
    r"<Ref\.[A-Za-z0-9]+>|\bRef(?:erence)?(?:\s*code)?\s*[:.=]?\s*[A-Za-z0-9]{3,8}\b",
    re.IGNORECASE,
)
REF_TH_RE = re.compile(r"(รหัสอ้างอิง[^\dA-Za-z\n]{0,10})[A-Za-z0-9]{4,8}")
PHONE_RE = re.compile(r"\b0\d{1,2}[-\s]?\d{3,4}[-\s]?\d{3,4}\b|\b0\d{2}-[xX]{3}-\d{4}\b")
AMOUNT_RE = re.compile(r"\d[\d,]*(?:\.\d+)?\s*(?:บาท|บ\.|บ(?=\s|/|$|\))|\.-)")


In [4]:
def normalize_sms(text: str) -> str:
    s = re.sub(r"\s+", " ", str(text).replace("\xa0", " ")).strip()
    s = URL_RE.sub("<URL>", s)
    s = LINE_RE.sub("<LINE_ID>", s)
    s = USSD_RE.sub("<USSD>", s)
    s = DATE_RE.sub("<DATE>", s)
    s = TIME_RE.sub("<TIME>", s)
    s = OTP_RE.sub("<OTP>", s)
    s = OTP_TH_RE.sub(r"\1<OTP>", s)
    s = REF_RE.sub("<REF>", s)
    s = REF_TH_RE.sub(r"\1<REF>", s)
    s = PHONE_RE.sub("<PHONE>", s)
    s = AMOUNT_RE.sub(" <AMOUNT> ", s)
    s = re.sub(r"\s*(<[A-Z_]+>)\s*", r" \1 ", s)
    return re.sub(r"\s+", " ", s).strip()

df["sentence"] = df["sentence_raw"].apply(normalize_sms)

# แสดงตัวอย่างเปรียบเทียบก่อนและหลังแปลง <TAG>
sample_df = df[["sentence_raw", "sentence", "group", "sub_group"]].sample(10, random_state=42)
for i, row in sample_df.iterrows():
    print(f"[{row['group']} | {row['sub_group']}]")
    print(f"  RAW : {row['sentence_raw']}")
    print(f"  TAG : {row['sentence']}\n")

display(sample_df)

[positive | carrier_vas]
  RAW : เสริมพลังสิริมงคล! แป้นพิมพ์มือถือลายพระแม่ธรณีสีเขียวมรกต ราคาพิเศษ!ลูกค้าเอไอเอสเพียง 69บาทเท่านั้น กด *424# โทรออก มีธีมสายมูราคาดีให้เลือกอีกเพียบ!
  TAG : เสริมพลังสิริมงคล! แป้นพิมพ์มือถือลายพระแม่ธรณีสีเขียวมรกต ราคาพิเศษ!ลูกค้าเอไอเอสเพียง <AMOUNT> เท่านั้น กด <USSD> โทรออก มีธีมสายมูราคาดีให้เลือกอีกเพียบ!

[positive | promotion]
  RAW : ช้อป Lazada ตีลเต็ตลดด่วน ตีลลดสูงสุด90% คลิก <URL>
  TAG : ช้อป Lazada ตีลเต็ตลดด่วน ตีลลดสูงสุด90% คลิก <URL>

[negative | phishing_link]
  RAW : FlashExpress ติดต่อคุณไม่ได้ กรุณายืนยันพัสดุของคุณอีกครั้ง : cutt.ly/FlahExpress
  TAG : FlashExpress ติดต่อคุณไม่ได้ กรุณายืนยันพัสดุของคุณอีกครั้ง : <URL>

[positive | carrier_vas]
  RAW : เพลงรอสายใหม่! นอนไม่หลับ-MEYOU, หยุดรัก-Zom Marie x Daou Pittaya ฟรี!ค่าบริการ30วัน สมัครโทร.*789*2# หรือที่เว็บ AIS CallingMelody
  TAG : เพลงรอสายใหม่! นอนไม่หลับ-MEYOU, หยุดรัก-Zom Marie x Daou Pittaya ฟรี!ค่าบริการ30วัน สมัครโทร. <USSD> หรือที่เว็บ AIS CallingMelody

[nega

,sentence_raw,sentence,group,sub_group
1023,เสริมพลังสิริมงคล! แป้นพิมพ์มือถือลายพระแม่ธรณ...,เสริมพลังสิริมงคล! แป้นพิมพ์มือถือลายพระแม่ธรณ...,positive,carrier_vas
381,ช้อป Lazada ตีลเต็ตลดด่วน ตีลลดสูงสุด90% คลิก ...,ช้อป Lazada ตีลเต็ตลดด่วน ตีลลดสูงสุด90% คลิก ...,positive,promotion
843,FlashExpress ติดต่อคุณไม่ได้ กรุณายืนยันพัสดุข...,FlashExpress ติดต่อคุณไม่ได้ กรุณายืนยันพัสดุข...,negative,phishing_link
427,"เพลงรอสายใหม่! นอนไม่หลับ-MEYOU, หยุดรัก-Zom M...","เพลงรอสายใหม่! นอนไม่หลับ-MEYOU, หยุดรัก-Zom M...",positive,carrier_vas
192,สล็อตเกมยิงปลา ฝาก 198 รับ 198 <URL>,สล็อตเกมยิงปลา ฝาก 198 รับ 198 <URL>,negative,invite
184,เครดิตฟรีรับง่าย ไม่ต้องแชร์ แค๊ปรูปมารับได้เล...,เครดิตฟรีรับง่าย ไม่ต้องแชร์ แค๊ปรูปมารับได้เล...,negative,invite
324,ซื้อประกันชีวิตอาวุโสโอเคให้พ่อแม่คุ้มครองชีวิ...,ซื้อประกันชีวิตอาวุโสโอเคให้พ่อแม่คุ้มครองชีวิ...,positive,promotion
49,ฉุกเฉินเงินไม่พอ ยืมจากทรูไปใช้ก่อนได้ ยืมเน็ต...,ฉุกเฉินเงินไม่พอ ยืมจากทรูไปใช้ก่อนได้ ยืมเน็ต...,positive,promotion
493,จัดอันดับเว็บแตกง่าย โบนัสเยอะ ฝากถอนออโต้ รัu...,จัดอันดับเว็บแตกง่าย โบนัสเยอะ ฝากถอนออโต้ รัu...,negative,invite
259,"คุณได้รับสินเชื่อ BAAC ธกส. 80,000 บาท สนใจคลิ...",คุณได้รับสินเชื่อ BAAC ธกส. <AMOUNT> สนใจคลิก ...,negative,invite


### Tokenization แบ่งคำโดยรักษาความหมายของประโยค
- ตัดคำภาษาไทยจาก `df["sentence"]` (เช่น ใช้ `pythainlp.word_tokenize` ด้วย engine `newmm` หรือ `attacut`)
- **ข้อควรระวัง:** ตั้งค่าหรือจัดการไม่ให้ `<TAG>` (เช่น `<URL>`, `<AMOUNT>`, `<OTP>`) ถูกตัดแยกเป็นคำย่อย และลองทดลองว่าจะตัด Stopwords หรือคงไว้

In [ ]:
LOOKALIKES = {
    "II": ["แ"], "ll": ["แ"], "Il": ["แ"], "lI": ["แ"],
    "I":  ["เ"], "l":  ["เ"], "|":  ["เ"],
    "o":  ["อ"], "O":  ["อ"], "0":  ["อ"],
    "u":  ["น", "บ"], "U": ["บ", "น"],
    "n":  ["ก", "ท"],
    "s":  ["ร", "ธ"], "S": ["ร", "ธ"],
    "w":  ["พ", "ฟ", "ฬ"], "W": ["ฟ", "พ", "ฬ"],
}

_UNIT        = re.compile("|".join(map(re.escape, sorted(LOOKALIKES, key=len, reverse=True))))
_CHUNK       = re.compile(rf"(?<![A-Za-z0-9])(?:[ก-๙]*(?:{_UNIT.pattern}))+[ก-๙]+(?![A-Za-z0-9])")
_ZERO_WIDTH  = re.compile(r"[\u200b-\u200d\u2060\ufeff]")
_PAREN       = re.compile(r"([\(\)])")
_FREQ        = dict(word_freqs())
_LOG_TOTAL   = math.log(sum(_FREQ.values()))
_DICT        = set(thai_words())
STOPWORDS    = thai_stopwords()
_NEGATIONS = {"ไม่", "ไม่ได้", "ไม่มี", "ไม่ใช่", "ห้าม", "อย่า", "มิ", "มิได้", "มิใช่", "ไม่ค่อย"}


@lru_cache(maxsize=200_000)
def _word_cost(word: str) -> float:
    return _LOG_TOTAL - math.log(_FREQ.get(word, 1 if word in _DICT else 0.01))

def _cost(text: str) -> float:
    return sum(_word_cost(w) for w in word_tokenize(text, keep_whitespace=False))

@lru_cache(maxsize=None)
def _fix_chunk(chunk: str, max_units: int = 3, min_gain: float = 5.0) -> str:
    parts, units = _UNIT.split(chunk), _UNIT.findall(chunk)
    if len(units) > max_units:
        return chunk
    best, best_cost = chunk, _cost(chunk) - min_gain
    for choice in itertools.product(*(LOOKALIKES[u] for u in units)):
        cand = parts[0] + "".join(c + p for c, p in zip(choice, parts[1:]))
        if (c := _cost(cand)) < best_cost:
            best, best_cost = cand, c
    return best

def preprocess(text: str) -> tuple[str, bool]:
    t = unicodedata.normalize("NFKC", str(text)).replace("\u0e4d\u0e32", "\u0e33")
    t = _ZERO_WIDTH.sub("", t)
    fixed          = _CHUNK.sub(lambda m: _fix_chunk(m.group()), t)
    was_obfuscated = fixed != t
    fixed          = re.sub(r"\s+", " ", _PAREN.sub(r" \1 ", fixed)).strip()
    return th_normalize(fixed), was_obfuscated

def tokenize(text: str, engine: str = "newmm") -> list[str]:
    return word_tokenize(text, engine=engine, keep_whitespace=False)

def remove_stopwords(tokens: list[str]) -> list[str]:
    result = []
    for i, t in enumerate(tokens):
        if t in STOPWORDS and t not in _NEGATIONS:
            if (i == 0) or (tokens[i - 1] not in _NEGATIONS):
                continue
        result.append(t)
    return result

In [ ]:
df[["sentence_clean", "has_obfuscation"]] = pd.DataFrame(df["sentence"].apply(preprocess).tolist(), index=df.index)
df["has_obfuscation"] = df["has_obfuscation"].astype(int)
df["tokens"]          = df["sentence_clean"].apply(tokenize)
df["tokens_nostop"]   = df["tokens"].apply(remove_stopwords)

for _, row in df.sample(5, random_state=42).iterrows():
    print(f"ORIGINAL  : {row['sentence']}")
    print(f"CLEAN     : {row['sentence_clean']}")
    print(f"TOKENS    : {row['tokens']}")
    print(f"STOPWORD   : {row['tokens_nostop']}")
    print(f"OBFUSCATED: {row['has_obfuscation']}\n") 

### แบ่งข้อมูล Train / Test (Train-Test Split)
- แบ่งข้อมูลที่ตัดคำแล้วพร้อม label (`group`, `sub_group`) เป็นชุด Train (เช่น 80%) และ Test (เช่น 20%) โดยใช้ `stratify` เพื่อรักษาสัดส่วนคลาส
- **หมายเหตุ:** ต้องแบ่งข้อมูลก่อนขั้นตอน Vectorize เพื่อป้องกัน Data Leakage

In [ ]:
def split_data(df, test_size=0.2, random_state=42) -> dict:
    X_train, X_test, y_train, y_test = train_test_split(
        df["tokens"],
        df[["group", "sub_group"]],
        test_size=test_size,
        random_state=random_state,
        stratify=df["sub_group"]
    )
    return {"X_train": X_train, "X_test": X_test,
            "y_train": y_train, "y_test": y_test}

splits = split_data(df)
X_train, X_test = splits["X_train"], splits["X_test"]
y_train, y_test = splits["y_train"], splits["y_test"]

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"\nสัดส่วน sub_group ใน Train:")
print(y_train["sub_group"].value_counts())
print(f"\nสัดส่วน sub_group ใน Test:")
print(y_test["sub_group"].value_counts())

### Vectorization
แปลงข้อความที่ตัดคำแล้วให้อยู่ในรูป Vector เพื่อส่งต่อให้โมเดล ML *(เลือกทดลองเปรียบเทียบวิธีที่เหมาะสมที่สุดได้อิสระ)*:
- `TfidfVectorizer` (ลองปรับ `ngram_range=(1, 2)`, `min_df`, `sublinear_tf=True`)
- `CountVectorizer` (Bag of Words)
- Word Embedding / Pre-trained Model (เช่น `Word2Vec`, `FastText`, หรือ `WangchanBERTa`)
- **ข้อสำคัญ:** ใช้ `fit_transform()` กับ **Train Set** เท่านั้น ส่วน Test Set ใช้ `.transform()`

In [ ]:
def build_vectorizer() -> TfidfVectorizer:
    return TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
        max_df=0.9,
        max_features=5000,
        token_pattern=r"(?u)\S+",
    )

vectorizer = build_vectorizer()

X_train_vec = vectorizer.fit_transform(X_train.apply(" ".join))
X_test_vec  = vectorizer.transform(X_test.apply(" ".join))

print(f"Shape Train: {X_train_vec.shape}")
print(f"Shape Test : {X_test_vec.shape}")
print(f"จำนวน features (คำ/bigram): {len(vectorizer.get_feature_names_out())}")
print(f"\nตัวอย่าง features 50 อันแรก:")
print(vectorizer.get_feature_names_out()[:50])

---
## ML

### 1. สร้างและเปรียบเทียบ Classification Model
รับ Vector จากส่วน NLP มาเทรนและเปรียบเทียบอัลกอริทึมเพื่อหาตัวที่แม่นยำและเร็วที่สุดสำหรับทั้ง `group` (`positive` / `negative`) และ `sub_group` (8 หมวดหมู่):
- **อัลกอริทึมแนะนำให้ลองเปรียบเทียบ (ปรับเปลี่ยนหรือเพิ่มได้อิสระ):**
  - **Naive Bayes:** `MultinomialNB`, `ComplementNB` *(เหมาะกับงานข้อความ/TF-IDF และเร็วมาก)*
  - **Linear Models:** `LogisticRegression`, `LinearSVC` / `SVM` *(มักให้ความแม่นยำสูงกับข้อความภาษาไทย)*
  - **Ensemble / Tree-based:** `RandomForestClassifier`, `XGBoost` / `LightGBM`, `VotingClassifier`
- *คำแนะนำ:* ลองทดสอบ `class_weight="balanced"` กับคลาสที่มีข้อมูลน้อย (เช่น `threat`, `payment_alert`)

### 2. วัดผลและปรับจูนโมเดล (Evaluation & Hyperparameter Tuning)
- ประเมินผลบน **Test Set** ด้วย `Accuracy`, `Precision`, `Recall`, `F1-Score` และ `Confusion Matrix` *(ควรดู F1-Score และ Recall ของฝั่ง `negative` เป็นพิเศษ เพื่อลดการหลุดของ SMS มิจฉาชีพ)*
- ปรับจูน Hyperparameter (เช่น `GridSearchCV` / `RandomizedSearchCV` หรือ `cross_val_score`) เพื่อเลือกโมเดลที่ดีที่สุดและใช้เวลาประมวลผลสมเหตุสมผล

### 3. ทดสอบกับ SMS ดิบจริง & บันทึกโมเดลสำหรับ FastAPI (Inference & Export)
- ทดลองนำข้อความ SMS ดิบใหม่ๆ มาผ่าน `normalize_sms()` -> Tokenize -> Vectorizer -> Model เพื่อดูผลทำนายและค่าความมั่นใจ (`predict_proba` / `decision_function`)
- บันทึก Vectorizer และ Model ที่ดีที่สุดด้วย `joblib` หรือ `pickle` (เช่น `vectorizer.joblib`, `model_group.joblib`, `model_subgroup.joblib`) เพื่อส่งต่อให้ฝั่ง FastAPI โหลดไปใช้งาน